In [1]:
from pathlib import Path
import sys

SCENARIOS = "base_run_EV"
NAME = "BL_8K"

###########################################################################
netcdf_filename = "output/Final_Set/" + NAME + ".nc"
yaml_filename = "output/Final_Set/" + NAME + ".yaml"

flow_csv_filename = "analysis_results/2.Scenario_S1/flow_with_utilization_and_coords.csv"
geojson_filename = "Research_Runs/Postprocessing/NUTS2.geojson"
output_html_path = "analysis_results/Final_Set/Installed_Capacities"+NAME+".html"

html_title = "Installed Capacities - "  + NAME

postproc_dir = Path.cwd() / "Research_Runs" / "Postprocessing"
sys.path.append(str(postproc_dir))

# Run installed capacity visualization
from postprocessing import run_IC
run_IC(
    netcdf_path=netcdf_filename,
    model_yaml_path=yaml_filename,
    output_html_path=output_html_path,
    html_title=html_title
)


Combined page saved to analysis_results/Final_Set/Installed_CapacitiesBL_8K.html


In [ ]:
from pathlib import Path
import sys


###########################################################################
# Define paths
output_folder = Path("output/Flows")
flow_csv_filename = "analysis_results/Final_Set/flow_with_utilization_and_coords.csv"
geojson_filename = "Research_Runs/Postprocessing/NUTS2.geojson"

postproc_dir = Path.cwd() / "Research_Runs" / "Postprocessing"
sys.path.append(str(postproc_dir))

# Import the visualization function
from postprocessing import run_IC


# Find all .nc files in the Final_Set folder
nc_files = list(output_folder.glob("*.nc"))

print(f"Found {len(nc_files)} .nc files to process")

# Process each .nc file
for nc_file in nc_files:
    # Extract the base name (without extension)
    NAME = nc_file.stem
    
    # Check if corresponding .yaml file exists
    yaml_file = output_folder / f"{NAME}.yaml"
    
    if not yaml_file.exists():
        print(f"Warning: No matching .yaml file found for {NAME}, skipping...")
        continue
    
    # Determine SCENARIOS based on filename
    if "KM" in NAME:
        SCENARIOS = "base_run_KM"
    else:
        SCENARIOS = "base_run_EV"
    
    # Define file paths
    netcdf_filename = str(nc_file)
    yaml_filename = str(yaml_file)
    output_html_path = f"analysis_results/Final_Set/Installed_Capacities_{NAME}_2.html"
    html_title = f"Installed Capacities - {NAME}"
    
    print(f"\nProcessing: {NAME}")
    print(f"  SCENARIOS: {SCENARIOS}")
    print(f"  NetCDF: {netcdf_filename}")
    print(f"  YAML: {yaml_filename}")
    print(f"  Output: {output_html_path}")
    
    try:
        # Run installed capacity visualization
        run_IC(
            netcdf_path=netcdf_filename,
            model_yaml_path=yaml_filename,
            output_html_path=output_html_path,
            html_title=html_title
        )
        print(f"  ✓ Successfully processed {NAME}")
    except Exception as e:
        print(f"  ✗ Error processing {NAME}: {str(e)}")
        continue

print("\n" + "="*50)
print("Processing complete!")


Found 9 .nc files to process

Processing: BL_H5_EV
  SCENARIOS: base_run_EV
  NetCDF: output/Final_Set/BL2/BL_H5_EV.nc
  YAML: output/Final_Set/BL2/BL_H5_EV.yaml
  Output: analysis_results/Final_Set/Installed_Capacities_BL_H5_EV_2.html
Combined page saved to analysis_results/Final_Set/Installed_Capacities_BL_H5_EV_2.html
  ✓ Successfully processed BL_H5_EV

Processing: BL_EV
  SCENARIOS: base_run_EV
  NetCDF: output/Final_Set/BL2/BL_EV.nc
  YAML: output/Final_Set/BL2/BL_EV.yaml
  Output: analysis_results/Final_Set/Installed_Capacities_BL_EV_2.html
Combined page saved to analysis_results/Final_Set/Installed_Capacities_BL_EV_2.html
  ✓ Successfully processed BL_EV

Processing: BL_12K
  SCENARIOS: base_run_EV
  NetCDF: output/Final_Set/BL2/BL_12K.nc
  YAML: output/Final_Set/BL2/BL_12K.yaml
  Output: analysis_results/Final_Set/Installed_Capacities_BL_12K_2.html
Combined page saved to analysis_results/Final_Set/Installed_Capacities_BL_12K_2.html
  ✓ Successfully processed BL_12K

Processing

In [11]:
# Jupyter cell: consolidate Calliope v0.6 run summaries into one CSV

import xarray as xr
import pandas as pd
import yaml
from pathlib import Path

# Directory containing all runs
runs_dir = Path.cwd() / "output" / "Final_Set" / "BL2"
run_ids = sorted({f.stem for f in runs_dir.glob("*.nc")})

# Define function to extract cost summaries
def cost_totals(netcdf_path):
    ds = xr.open_dataset(netcdf_path)
    # monetary slices
    var_da = ds["cost_var"].sel(costs="monetary")              # loc_techs_om_cost × timesteps
    inv_da = ds["cost_investment"].sel(costs="monetary")       # loc_techs_investment_cost

    # parse loc::tech identifiers
    loc_var = ds.coords["loc_techs_om_cost"].values
    tech_var = [lt.split("::", 1)[1] for lt in loc_var]
    loc_inv = ds.coords["loc_techs_investment_cost"].values
    tech_inv = [lt.split("::", 1)[1] for lt in loc_inv]

    # aggregate per-tech
    df_var = (
        pd.DataFrame({
            "tech": tech_var,
            "var_cost": var_da.sum(dim="timesteps").values
        })
        .groupby("tech")["var_cost"]
        .sum()
    )
    df_inv = (
        pd.DataFrame({
            "tech": tech_inv,
            "fixed_cost": inv_da.values
        })
        .groupby("tech")["fixed_cost"]
        .sum()
    )
    df = pd.concat([df_var, df_inv], axis=1).fillna(0)

    # masks
    mask_tech    = ~df.index.str.contains("transmission|interconnector|^ES_")
    mask_infra   = df.index.str.contains("transmission|interconnector")
    mask_storage = df.index.str.startswith("ES_")

    # compute
    total_tech_cost_fixed    = df.loc[mask_tech, "fixed_cost"].sum()
    total_tech_cost_var      = df.loc[mask_tech, "var_cost"].sum()
    total_infra_cost_fixed   = df.loc[mask_infra, "fixed_cost"].sum()
    total_infra_cost_var     = df.loc[mask_infra, "var_cost"].sum()
    total_storage_cost_fixed = df.loc[mask_storage, "fixed_cost"].sum()
    total_storage_cost_var   = df.loc[mask_storage, "var_cost"].sum()
    total_var_cost           = df["var_cost"].sum()
    total_fixed_cost         = df["fixed_cost"].sum()

    return {
        "total_tech_cost_fixed_M€": total_tech_cost_fixed,
        "total_tech_cost_var_M€":   total_tech_cost_var,
        "total_infra_cost_fixed_M€": total_infra_cost_fixed,
        "total_infra_cost_var_M€":   total_infra_cost_var,
        "total_storage_cost_fixed_M€": total_storage_cost_fixed,
        "total_storage_cost_var_M€":   total_storage_cost_var,
        "total_var_cost_M€":           total_var_cost,
        "total_fixed_cost_M€":         total_fixed_cost,
        "total_system_cost_M€":        total_var_cost + total_fixed_cost,
    }

# List of all SMR tech IDs
SMR_TECH_IDS = [
    "pp_SMR_Hitachi_CHP_1", "pp_SMR_RollsRoyce_CHP_1", "pp_SMR_Thorizon_CHP_1",
    "pp_SMR_Hitachi_CH2P_1",  "pp_SMR_RollsRoyce_CH2P_1",  "pp_SMR_Thorizon_CH2P_1",
    "pp_SMR_Hitachi_CHH2P_1", "pp_SMR_RollsRoyce_CHH2P_1", "pp_SMR_Thorizon_CHH2P_1",
    "pp_SMR_Hitachi_CHP_2", "pp_SMR_RollsRoyce_CHP_2", "pp_SMR_Thorizon_CHP_2",
    "pp_SMR_Hitachi_CH2P_2",  "pp_SMR_RollsRoyce_CH2P_2",  "pp_SMR_Thorizon_CH2P_2",
    "pp_SMR_Hitachi_CHH2P_2", "pp_SMR_RollsRoyce_CHH2P_2", "pp_SMR_Thorizon_CHH2P_2",
    "pp_SMR_Hitachi_CHP_3", "pp_SMR_RollsRoyce_CHP_3", "pp_SMR_Thorizon_CHP_3",
    "pp_SMR_Hitachi_CH2P_3",  "pp_SMR_RollsRoyce_CH2P_3",  "pp_SMR_Thorizon_CH2P_3",
    "pp_SMR_Hitachi_CHH2P_3", "pp_SMR_RollsRoyce_CHH2P_3", "pp_SMR_Thorizon_CHH2P_3"
]

# Function to count SMR units from YAML
def count_smr_units(yaml_path):
    try:
        cfg = yaml.safe_load(yaml_path.read_text())
        count = 0
        for tech_id, tech_cfg in cfg.get("techs", {}).items():
            if tech_id in SMR_TECH_IDS:
                count += tech_cfg.get("units", 0)
        return count
    except Exception as e:
        return f"error:{e}"

# Iterate over runs and build summary table
records = []
for run in run_ids:
    row = {"run_id": run}
    nc_path = runs_dir / f"{run}.nc"
    yaml_path = runs_dir / f"{run}.yaml"
    # cost summaries
    try:
        costs = cost_totals(nc_path)
    except Exception as e:
        costs = {k: "N/A" for k in [
            "total_tech_cost_fixed_M€", "total_tech_cost_var_M€",
            "total_infra_cost_fixed_M€", "total_infra_cost_var_M€",
            "total_storage_cost_fixed_M€", "total_storage_cost_var_M€",
            "total_var_cost_M€", "total_fixed_cost_M€", "total_system_cost_M€"
        ]}
        row["error_costs"] = str(e)
    row.update(costs)

    # SMR count
    row["n_smr_units"] = count_smr_units(yaml_path)

    records.append(row)

# Build DataFrame and export
df_summary = pd.DataFrame.from_records(records).set_index("run_id")
df_summary.to_csv("Final_Set_Costs_BL2.csv")
df_summary.head()


,total_tech_cost_fixed_M€,total_tech_cost_var_M€,total_infra_cost_fixed_M€,total_infra_cost_var_M€,total_storage_cost_fixed_M€,total_storage_cost_var_M€,total_var_cost_M€,total_fixed_cost_M€,total_system_cost_M€,n_smr_units
run_id,,,,,,,,,,
BL_12K,17285.056038,7300.590124,2873.096306,13032.741026,3433.980709,0.0,20333.331150,23592.133054,43925.464204,0
BL_8K,17224.669610,7285.170661,2869.731532,13162.291177,3438.575596,0.0,20447.461838,23532.976737,43980.438575,0
BL_EV,17291.591805,7495.504221,2893.220065,13151.883452,3399.630791,0.0,20647.387673,23584.442661,44231.830334,0
BL_H2_EV,17387.079132,7516.405842,2903.837427,13784.768095,3369.957873,0.0,21301.173936,23660.874433,44962.048369,0
BL_H2_KM,15642.043269,12943.799069,2899.509657,2787.922165,2730.962377,0.0,15731.721234,21272.515303,37004.236538,0


In [12]:
# Jupyter cell: combined displacement capacity CSV for all Calliope runs (filtered techs, include SMRs)

import xarray as xr
import pandas as pd
from pathlib import Path

# Directory containing all runs
runs_dir = Path.cwd() / "output" / "Final_Set" / "BL2"
run_ids = sorted({f.stem for f in runs_dir.glob("*.nc")})

# Tech prefixes to exclude
exclude_prefixes = [
    "free_gas_transmission",
    "free_hyd_transmission",
    "free_co2_stored_transmission",
    "free_solid_fuel_transmission",
    "free_ets_budget_transmission",
    "free_ets_penalty_transmission",
    "free_sink_transmission",
    "free_co2_transmission",
]

all_records = []

for run in run_ids:
    nc_path = runs_dir / f"{run}.nc"
    try:
        ds = xr.open_dataset(nc_path)
        # Extract installed energy capacity (GW) per loc::tech
        cap_da = ds["energy_cap"]  # dims: loc_techs
        loc_techs = ds.coords["loc_techs"].values
        capacities = cap_da.values

        # Split loc::tech identifiers
        locs = [lt.split("::", 1)[0] for lt in loc_techs]
        techs = [lt.split("::", 1)[1] for lt in loc_techs]

        # Build DataFrame
        df = pd.DataFrame({
            "loc": locs,
            "tech": techs,
            "capacity_GW": capacities
        })

        # Filter out excluded techs
        mask_exclude = df["tech"].apply(lambda t: any(t.startswith(pref) for pref in exclude_prefixes))
        df = df[~mask_exclude]

        # Pivot to loc × tech table
        pivot = df.pivot_table(
            index="loc",
            columns="tech",
            values="capacity_GW",
            aggfunc="sum",
            fill_value=0
        )

        # Add total row and column
        pivot.loc["Total"] = pivot.sum(axis=0)
        pivot["Total"] = pivot.sum(axis=1)

        # Flatten pivot for concatenation
        pivot = pivot.reset_index()
        pivot.insert(0, "run_id", run)

        all_records.append(pivot)

    except Exception as e:
        # placeholder error row
        err_df = pd.DataFrame({
            "run_id": [run],
            "loc": ["error"],
        })
        all_records.append(err_df)

# Concatenate all runs into one DataFrame and save
df_all = pd.concat(all_records, ignore_index=True, sort=False)
df_all.to_csv("Final_Set_Capacities_BL2.csv", index=False)
df_all.head()


tech,run_id,loc,ES_BESS_IDES,ES_BESS_MDES,ES_BESS_households,co2_emissions_sink,co2_storage,demand_elc,demand_gas,demand_hyd,...,transmission_hvac:E-VHZ,transmission_hvac:E-VVL,transmission_hvac:E-WES,transmission_hvac:E-WEW,transmission_hvac:E-WTR,transmission_hvac:E-ZWO,transmission_hvac:E-ZYV,wind_offshore,wind_onshore,Total
0,BL_12K,E-BE-EYC,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,3.600000
1,BL_12K,E-BE-ZAN,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,7.050000
2,BL_12K,E-BGM,0.200722,0.000000,0.019486,0.0,0.0,0.265307,0.114069,0.026035,...,0.0,1.905256,0.0,0.0,0.0,0.0,0.0,0.0,0.060301,5.807192
3,BL_12K,E-BMR,0.383377,0.028719,0.038107,0.0,0.0,0.645390,0.168178,0.068266,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.138010,10.529116
4,BL_12K,E-BRK,0.251755,0.000000,0.074316,0.0,0.0,1.674418,0.459840,0.019441,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.623515,11.937685


Get Curtailment

inspect single run

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# Load clustered results
ds = xr.open_dataset(Path("output/Test_last/HIX_C150.nc"))

# Technologies to analyze
techs = ['wind_offshore', 'wind_onshore', 'pv_utility', 'pv_rooftop']

# Timestep weights (hours per representative timestep)
w = ds['timestep_weights']

records = []
for lt in ds.coords['loc_techs_finite_resource'].values:
    loc, tech = lt.split("::")
    if tech not in techs:
        continue

    # Installed capacity [GW]
    C = ds['energy_cap'].sel(loc_techs=f"{loc}::{tech}").item()
    if C <= 0:
        continue

    # Resource: GWh per GW per cluster
    r = ds['resource'].sel(loc_techs_finite_resource=lt)

    # Production: GWh per cluster
    prods = [
        ds['carrier_prod'].sel(loc_tech_carriers_prod=lcp)
        for lcp in ds.coords['loc_tech_carriers_prod'].values
        if lcp.startswith(f"{loc}::{tech}::")
    ]
    if not prods:
        continue
    p = sum(prods)

    # Available per cluster [GWh]
    avail = C * r

    # Curtailment per cluster [GWh]
    curt = (avail - p).clip(min=0)

    # Weight by cluster hours [h]
    curt_real = curt * w

    # Annual totals [GWh/year]
    A = (avail * w).sum().item()   # Available energy
    P = (p     * w).sum().item()   # Produced energy
    C_year = curt_real.sum().item()# Curtailed energy

    records.append({
        'location':       loc,
        'technology':     tech,
        'capacity_GW':    C,
        'available_GWh':  round(A,2),
        'produced_GWh':   round(P,2),
        'curtailed_GWh':  round(C_year,2),
        'curtailment_%':  round(C_year/A*100,2) if A>0 else np.nan
    })

df = pd.DataFrame(records).sort_values('curtailed_GWh', ascending=False)

print("Curtailment by location::tech")
print(df.to_string(index=False))

summary = df.groupby('technology')[['available_GWh','produced_GWh','curtailed_GWh']].sum()
summary['curtailment_%'] = (summary['curtailed_GWh']/summary['available_GWh']*100).round(4)

print("\nSummary by technology")
print(summary.to_string())

# Save results
df.to_csv("output/Test_last/RR_EV_curtailment.csv", index=False)
summary.to_csv("output/Test_last/RR_EV_curtailment_by_tech.csv")


Inspect single calculation

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# Load results
ds = xr.open_dataset(Path("output/Test_last/HIX_C150.nc"))

loc_tech = "E-CST::wind_onshore"
loc, tech = loc_tech.split("::")

C = ds["energy_cap"].sel(loc_techs=loc_tech).item()            # GW
r = ds["resource"].sel(loc_techs_finite_resource=loc_tech)     # GWh/GW per cluster
prod = sum(
    ds["carrier_prod"].sel(loc_tech_carriers_prod=lcp)
    for lcp in ds.coords["loc_tech_carriers_prod"].values
    if lcp.startswith(f"{loc_tech}::")
)                                                               # GWh per cluster
w = ds["timestep_weights"]                                     # hours per cluster

# Epsilon tolerance (GWh)
eps = 1e-3

df = pd.DataFrame({
    "timestep":          ds.coords["timesteps"].values,
    "cluster_weight_h":  w.values,
    "energy_cap_GW":     C,
    "energy_per_cap":    r.values,
})
df["available_GWh"] = df["energy_per_cap"] * df["energy_cap_GW"]
df["electricity_prod"] = prod.values

# Compute curtailment with tolerance
df["raw_diff"] = df["available_GWh"] - df["electricity_prod"]
df["curtailed_GWh"] = df["raw_diff"].apply(lambda x: x if x > eps else 0.0)
df["curtailment_%"] = np.where(
    df["available_GWh"] > eps,
    df["curtailed_GWh"] / df["available_GWh"] * 100,
    np.nan
)

# Display with full precision
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:0.8f}".format)
df[[
    "timestep",
    "cluster_weight_h",
    "energy_cap_GW",
    "energy_per_cap",
    "available_GWh",
    "electricity_prod",
    "raw_diff",
    "curtailed_GWh",
    "curtailment_%",
]]


Get all csvs

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# Folder containing .nc files
nc_dir = Path("output/Final_Set/BL2")
# Technologies to analyze
techs = ['wind_offshore', 'wind_onshore', 'pv_utility', 'pv_rooftop']
# Tolerance for numerical errors (GWh)
eps = 1e-3

# Loop through all .nc files, skipping those with 'BL' in the name
for nc_file in nc_dir.glob("*.nc"):
    if "BL" in nc_file.name:
        continue

    ds = xr.open_dataset(nc_file)
    records = []

    # Cluster weights (hours per cluster)
    w = ds["timestep_weights"].values

    for loc_tech in ds.coords["loc_techs"].values:
        loc, tech = loc_tech.split("::")
        if tech not in techs:
            continue

        # Installed capacity (GW)
        C = ds["energy_cap"].sel(loc_techs=loc_tech).item()
        if C <= 0:
            continue

        # Resource (GWh/GW per cluster)
        r = ds["resource"].sel(loc_techs_finite_resource=loc_tech).values

        # Production (GWh per cluster)
        prod_terms = [
            ds["carrier_prod"].sel(loc_tech_carriers_prod=lcp).values
            for lcp in ds.coords["loc_tech_carriers_prod"].values
            if lcp.startswith(f"{loc_tech}::")
        ]
        if not prod_terms:
            continue
        p = sum(prod_terms)

        # Available energy per cluster [GWh]
        avail = r * C

        # Curtailment per cluster [GWh]
        raw_diff = avail - p
        curt = np.where(raw_diff > eps, raw_diff, 0.0)

        # Annual totals (weighted by cluster hours)
        A = np.sum(avail * w)
        P = np.sum(p * w)
        C_year = np.sum(curt * w)
        rate = (C_year / A * 100) if A > 0 else np.nan

        records.append({
            "loc_tech":        loc_tech,
            "energy_cap_GW":   C,
            "available_GWh":   round(A, 8),
            "produced_GWh":    round(P, 8),
            "raw_diff_GWh":    round(np.sum(raw_diff * w), 8),
            "curtailed_GWh":   round(C_year, 8),
            "curtailment_%":   round(rate, 8)
        })

    df = pd.DataFrame(records)
    # Save CSV with same base name as the .nc
    csv_file = nc_file.with_suffix("_BL2.csv")
    df.to_csv(csv_file, index=False)
    print(f"Processed {nc_file.name} → {csv_file.name}")

    ds.close()


Processed HI_EV_2.nc → HI_EV_2.csv
Processed HI_EV.nc → HI_EV.csv
Processed T3_KM.nc → T3_KM.csv
Processed RRX_EV.nc → RRX_EV.csv
Processed HIX12.nc → HIX12.csv
Processed T3_EV.nc → T3_EV.csv
Processed HI_C2.nc → HI_C2.csv
Processed RRX12.nc → RRX12.csv
Processed RRX_KM.nc → RRX_KM.csv


Create one single big overview

In [13]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path


# Folder containing .nc files
nc_dir = Path("output/Final_Set/BL2")
# Technologies to analyze
techs = ['wind_offshore', 'wind_onshore', 'pv_utility', 'pv_rooftop']
# Tolerance for numerical errors (GWh)
eps = 1e-3

# Store all runs' data
all_runs_data = []

# Loop through all .nc files, skipping those with 'BL' in the name
for nc_file in nc_dir.glob("*.nc"):

    print(f"Processing {nc_file.name}...")
    
    ds = xr.open_dataset(nc_file)
    records = []

    # Cluster weights (hours per cluster)
    w = ds["timestep_weights"].values

    for loc_tech in ds.coords["loc_techs"].values:
        loc, tech = loc_tech.split("::")
        if tech not in techs:
            continue

        # Installed capacity (GW)
        C = ds["energy_cap"].sel(loc_techs=loc_tech).item()
        if C <= 0:
            continue

        # Resource (GWh/GW per cluster)
        r = ds["resource"].sel(loc_techs_finite_resource=loc_tech).values

        # Production (GWh per cluster)
        prod_terms = [
            ds["carrier_prod"].sel(loc_tech_carriers_prod=lcp).values
            for lcp in ds.coords["loc_tech_carriers_prod"].values
            if lcp.startswith(f"{loc_tech}::")
        ]
        if not prod_terms:
            continue
        p = sum(prod_terms)

        # Available energy per cluster [GWh]
        avail = r * C

        # Curtailment per cluster [GWh]
        raw_diff = avail - p
        curt = np.where(raw_diff > eps, raw_diff, 0.0)

        # Annual totals (weighted by cluster hours)
        A = np.sum(avail * w)
        P = np.sum(p * w)
        C_year = np.sum(curt * w)
        rate = (C_year / A * 100) if A > 0 else 0.0

        records.append({
            "tech":            tech,
            "energy_cap_GW":   C,
            "available_GWh":   A,
            "produced_GWh":    P,
            "curtailed_GWh":   C_year,
            "curtailment_%":   rate
        })

    # Aggregate by technology for this run
    df = pd.DataFrame(records)
    
    # Initialize run data with RunID
    run_data = {"RunID": nc_file.stem}
    
    # For each technology, calculate the aggregated values
    for tech in techs:
        tech_data = df[df["tech"] == tech]
        
        if len(tech_data) > 0:
            # Sum of installed capacity
            total_cap = tech_data["energy_cap_GW"].sum()
            # Sum of curtailed energy
            total_curt_gwh = tech_data["curtailed_GWh"].sum()
            # Calculate overall curtailment percentage
            total_avail = tech_data["available_GWh"].sum()
            total_curt_pct = (total_curt_gwh / total_avail * 100) if total_avail > 0 else 0.0
        else:
            total_cap = 0.0
            total_curt_gwh = 0.0
            total_curt_pct = 0.0
        
        # Map tech names to readable column names
        tech_names = {
            'wind_offshore': 'Offshore Wind',
            'wind_onshore': 'Onshore Wind',
            'pv_utility': 'PV Utility',
            'pv_rooftop': 'PV Rooftop'
        }
        
        tech_label = tech_names[tech]
        run_data[f"{tech_label} TIC"] = round(total_cap, 4)
        run_data[f"{tech_label} Curtailment GWh"] = round(total_curt_gwh, 4)
        run_data[f"{tech_label} Curtailment %"] = round(total_curt_pct, 4)
    
    all_runs_data.append(run_data)
    ds.close()

# Create comprehensive dataframe with all runs
comprehensive_df = pd.DataFrame(all_runs_data)

# Define column order
column_order = ["RunID"]
for tech_label in ['Offshore Wind', 'Onshore Wind', 'PV Rooftop', 'PV Utility']:
    column_order.extend([
        f"{tech_label} TIC",
        f"{tech_label} Curtailment GWh",
        f"{tech_label} Curtailment %"
    ])

comprehensive_df = comprehensive_df[column_order]

# Save to CSV
output_file = nc_dir / "comprehensive_curtailment_overview_BL2.csv"
comprehensive_df.to_csv(output_file, index=False)
print(f"\nComprehensive overview saved to: {output_file}")
print(f"Total runs processed: {len(all_runs_data)}")


Processing BL_H5_EV.nc...
Processing BL_EV.nc...
Processing BL_12K.nc...
Processing BL_KM.nc...
Processing BL_8K.nc...
Processing BL_H5_KM.nc...
Processing BL_H2_KM.nc...
Processing BL_H2_EV.nc...
Processing BL_S2.nc...

Comprehensive overview saved to: output/Final_Set/BL2/comprehensive_curtailment_overview_BL2.csv
Total runs processed: 9


Get emissions

In [ ]:
import xarray as xr
import pandas as pd
from pathlib import Path

# Folder containing .nc files
nc_dir = Path("output/Final_Set")
output_dir = nc_dir / "emissions"
output_dir.mkdir(exist_ok=True)

def parse_loc_tech(s):
    parts = s.split("::")
    if len(parts) >= 2:
        return parts[0], parts[1]
    else:
        return s, ""

for nc_file in nc_dir.glob("*.nc"):
    ds = xr.open_dataset(nc_file)

    co2_emitted_prod = [
        c for c in ds.coords['loc_tech_carriers_prod'].values
        if 'co2_emitted' in c and 'transmission' not in c
    ]

    records = []
    for carrier in co2_emitted_prod:
        loc, tech = parse_loc_tech(carrier)
        total_co2 = ds['carrier_prod'].sel(loc_tech_carriers_prod=carrier).sum(dim='timesteps').item()
        records.append({
            'loc': loc,
            'tech': tech,
            'co2_emitted': round(total_co2, 8)
        })

    df = pd.DataFrame(records)

    csv_file = output_dir / nc_file.with_suffix(".csv").name
    df.to_csv(csv_file, index=False)
    print(f"Processed {nc_file.name} → emissions/{csv_file.name}")

    ds.close()


Processed HI_EV_2.nc → emissions/HI_EV_2.csv
Processed HI_EV.nc → emissions/HI_EV.csv
Processed T3_KM.nc → emissions/T3_KM.csv
Processed RRX_EV.nc → emissions/RRX_EV.csv
Processed HIX12.nc → emissions/HIX12.csv
Processed T3_EV.nc → emissions/T3_EV.csv
Processed HI_C2.nc → emissions/HI_C2.csv
Processed RRX12.nc → emissions/RRX12.csv
Processed RRX_KM.nc → emissions/RRX_KM.csv


1 big emissions file

In [14]:
import xarray as xr
import pandas as pd
from pathlib import Path


# Folder containing .nc files
nc_dir = Path("output/Final_Set/BL2")
output_dir = nc_dir / "emissions"
output_dir.mkdir(exist_ok=True)


def parse_loc_tech(s):
    parts = s.split("::")
    if len(parts) >= 2:
        return parts[0], parts[1]
    else:
        return s, ""


def format_tech_name(tech):
    """Convert tech name like 'pp_ccgt_gas' to 'Emissions CCGT Gas'"""
    # Remove common prefixes
    tech = tech.replace('pp_', '').replace('_', ' ')
    # Capitalize words
    words = tech.split()
    formatted = ' '.join(word.upper() if len(word) <= 4 else word.capitalize() for word in words)
    return f"Emissions {formatted}"


# Store all runs' emissions data
all_runs_emissions = []

for nc_file in nc_dir.glob("*.nc"):
    print(f"Processing {nc_file.name}...")
    
    ds = xr.open_dataset(nc_file)

    co2_emitted_prod = [
        c for c in ds.coords['loc_tech_carriers_prod'].values
        if 'co2_emitted' in c and 'transmission' not in c
    ]

    records = []
    for carrier in co2_emitted_prod:
        loc, tech = parse_loc_tech(carrier)
        total_co2 = ds['carrier_prod'].sel(loc_tech_carriers_prod=carrier).sum(dim='timesteps').item()
        records.append({
            'loc': loc,
            'tech': tech,
            'co2_emitted': round(total_co2, 8)
        })

    df = pd.DataFrame(records)

    # Save individual CSV (original functionality)
    csv_file = output_dir / nc_file.with_suffix(".csv").name
    df.to_csv(csv_file, index=False)
    print(f"  Individual file saved → emissions/{csv_file.name}")

    # Aggregate emissions by technology (sum across all locations)
    tech_emissions = df.groupby('tech')['co2_emitted'].sum().to_dict()
    
    # Create run data with RunID and emissions per tech
    run_data = {"RunID": nc_file.stem}
    
    for tech, emissions in tech_emissions.items():
        column_name = format_tech_name(tech)
        run_data[column_name] = round(emissions, 4)
    
    all_runs_emissions.append(run_data)
    
    ds.close()

# Create comprehensive emissions dataframe
comprehensive_emissions_df = pd.DataFrame(all_runs_emissions)

# Fill NaN values with 0 (for technologies not present in all runs)
comprehensive_emissions_df = comprehensive_emissions_df.fillna(0)

# Sort columns: RunID first, then all emissions columns alphabetically
emissions_columns = sorted([col for col in comprehensive_emissions_df.columns if col != "RunID"])
column_order = ["RunID"] + emissions_columns
comprehensive_emissions_df = comprehensive_emissions_df[column_order]

# Save comprehensive overview
overview_file = nc_dir / "comprehensive_emissions_overview_BL2.csv"
comprehensive_emissions_df.to_csv(overview_file, index=False)
print(f"\nComprehensive emissions overview saved to: {overview_file}")
print(f"Total runs processed: {len(all_runs_emissions)}")
print(f"Total unique technologies found: {len(emissions_columns)}")


Processing BL_H5_EV.nc...
  Individual file saved → emissions/BL_H5_EV.csv
Processing BL_EV.nc...
  Individual file saved → emissions/BL_EV.csv
Processing BL_12K.nc...
  Individual file saved → emissions/BL_12K.csv
Processing BL_KM.nc...
  Individual file saved → emissions/BL_KM.csv
Processing BL_8K.nc...
  Individual file saved → emissions/BL_8K.csv
Processing BL_H5_KM.nc...
  Individual file saved → emissions/BL_H5_KM.csv
Processing BL_H2_KM.nc...
  Individual file saved → emissions/BL_H2_KM.csv
Processing BL_H2_EV.nc...
  Individual file saved → emissions/BL_H2_EV.csv
Processing BL_S2.nc...
  Individual file saved → emissions/BL_S2.csv

Comprehensive emissions overview saved to: output/Final_Set/BL2/comprehensive_emissions_overview_BL2.csv
Total runs processed: 9
Total unique technologies found: 3


FLOWS

In [15]:
from flows_extraction import process_multiple_files_fast
from pathlib import Path

input_folder = Path("output/Final_Set/Remaining/BL2")            # folder with your .nc files
output_folder = Path("output/Final_Set/extracted_data")  # output for CSVs

results = process_multiple_files_fast(input_folder, output_folder, pattern="*.nc")



Found 4 .nc file(s) to process (fast)

Processing (fast): BL_H2_EV.nc
Extracting flows...
✓ Full tidy data saved: output/Final_Set/extracted_data/BL_H2_EV_full.csv
  Records: 1679040, Columns: 18
✓ Processing complete (fast)!


Processing (fast): BL_H2_KM.nc
Extracting flows...
✓ Full tidy data saved: output/Final_Set/extracted_data/BL_H2_KM_full.csv
  Records: 1679040, Columns: 18
✓ Processing complete (fast)!


Processing (fast): BL_H5_EV.nc
Extracting flows...
✓ Full tidy data saved: output/Final_Set/extracted_data/BL_H5_EV_full.csv
  Records: 1679040, Columns: 18
✓ Processing complete (fast)!


Processing (fast): BL_H5_KM.nc
Extracting flows...
✓ Full tidy data saved: output/Final_Set/extracted_data/BL_H5_KM_full.csv
  Records: 1679040, Columns: 18
✓ Processing complete (fast)!


PROCESSING SUMMARY (fast)
Successful: 4/4
  ✓ BL_H2_EV.nc: 1679040 records
  ✓ BL_H2_KM.nc: 1679040 records
  ✓ BL_H5_EV.nc: 1679040 records
  ✓ BL_H5_KM.nc: 1679040 records


In [8]:
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np

# Define your paths
nc_folder = Path("output/Final_Set")
output_folder = Path("output/Final_Set/extracted_data")

# Create output folder if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

# Find all .nc files (not in subfolders)
nc_files = sorted(list(nc_folder.glob("*.nc")))
print(f"Found {len(nc_files)} .nc files to process\n")

# List to store results
all_data = []

# Process each .nc file
for i, nc_file in enumerate(nc_files, 1):
    print(f"[{i}/{len(nc_files)}] Processing: {nc_file.name}")
    
    try:
        # Open the NetCDF file
        ds = xr.open_dataset(nc_file)
        
        # Extract timestep_weights
        timestep_weights = ds['timestep_weights'].values
        timesteps = ds['timesteps'].values
        
        # Create a DataFrame for this file
        df = pd.DataFrame({
            'nc_filename': nc_file.name,  # Just the filename, not full path
            'timesteps': timesteps,
            'weight_per_timestep': timestep_weights
        })
        
        all_data.append(df)
        
        ds.close()
        print(f"  ✓ Processed {len(timesteps)} timesteps")
        
    except Exception as e:
        print(f"  ✗ Error processing {nc_file.name}: {e}")
        continue

# Combine all data into one DataFrame
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Save to CSV
    output_csv = output_folder / "timestep_weights_all_models.csv"
    combined_df.to_csv(output_csv, index=False)
    
    print(f"\n✓ Successfully saved to: {output_csv}")
    print(f"\nTotal rows: {len(combined_df)}")
    print(f"Unique models: {combined_df['nc_filename'].nunique()}")
    print(f"\nFirst few rows:")
    print(combined_df.head(10))
    
else:
    print("\nNo data was processed!")


Found 59 .nc files to process

[1/59] Processing: BL_EV.nc
  ✓ Processed 240 timesteps
[2/59] Processing: BL_H2_EV.nc
  ✓ Processed 240 timesteps
[3/59] Processing: BL_H2_KM.nc
  ✓ Processed 240 timesteps
[4/59] Processing: BL_H5_EV.nc
  ✓ Processed 240 timesteps
[5/59] Processing: BL_H5_KM.nc
  ✓ Processed 240 timesteps
[6/59] Processing: BL_KM.nc
  ✓ Processed 240 timesteps
[7/59] Processing: HIX_C150_EV.nc
  ✓ Processed 240 timesteps
[8/59] Processing: HIX_C150_KM.nc
  ✓ Processed 240 timesteps
[9/59] Processing: HIX_C75_EV.nc
  ✓ Processed 240 timesteps
[10/59] Processing: HIX_C75_KM.nc
  ✓ Processed 240 timesteps
[11/59] Processing: HIX_EV.nc
  ✓ Processed 240 timesteps
[12/59] Processing: HIX_H2_EV.nc
  ✓ Processed 240 timesteps
[13/59] Processing: HIX_H2_KM.nc
  ✓ Processed 240 timesteps
[14/59] Processing: HIX_KM.nc
  ✓ Processed 240 timesteps
[15/59] Processing: HI_C150_KM.nc
  ✓ Processed 240 timesteps
[16/59] Processing: HI_C75_EV.nc
  ✓ Processed 240 timesteps
[17/59] Proce

In [14]:
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np


# Define your paths
nc_folder = Path("output/Final_Set/extracted_data")  # folder with your .nc files
output_folder = Path("output/Final_Set/extracted_data")  # output for CSV


# Find the first .nc file in the folder (not in subfolders)
nc_files = list(nc_folder.glob("*.nc"))
print(f"Found {len(nc_files)} .nc files")


if nc_files:
    # Test with the first file
    test_file = nc_files[0]
    print(f"\nTesting with: {test_file.name}")
    
    # Open the NetCDF file
    ds = xr.open_dataset(test_file)
    
    # Extract timestep_weights
    timestep_weights = ds['timestep_weights'].values
    
    # Get timesteps as strings (they might be datetime objects)
    timesteps = ds['timesteps'].values
    
    print(f"\nNumber of timesteps: {len(timesteps)}")
    print(f"First 5 timesteps: {timesteps[:5]}")
    print(f"First 5 weights: {timestep_weights[:5]}")
    
    # Create a DataFrame to see the structure
    df = pd.DataFrame({
        'timesteps': timesteps,
        'weight': timestep_weights
    })
    
    print("\nDataFrame head:")
    print(df.head(10))
    
    # Check the data types
    print(f"\nTimestep dtype: {df['timesteps'].dtype}")
    print(f"Weight dtype: {df['weight'].dtype}")
    
    # Save to CSV
    output_folder.mkdir(parents=True, exist_ok=True)
    output_csv = output_folder / test_file.name.replace('.nc', '_weights.csv')
    df.to_csv(output_csv, index=False)
    
    print(f"\n✓ CSV saved to: {output_csv}")
    print(f"✓ Total rows saved: {len(df)}")
    
    ds.close()
else:
    print("No .nc files found in the folder!")

Found 1 .nc files

Testing with: BL_H5_KM.nc

Number of timesteps: 240
First 5 timesteps: ['2040-09-26T00:00:00.000000000' '2040-09-26T01:00:00.000000000'
 '2040-09-26T02:00:00.000000000' '2040-09-26T03:00:00.000000000'
 '2040-09-26T04:00:00.000000000']
First 5 weights: [41. 41. 41. 41. 41.]

DataFrame head:
            timesteps  weight
0 2040-09-26 00:00:00    41.0
1 2040-09-26 01:00:00    41.0
2 2040-09-26 02:00:00    41.0
3 2040-09-26 03:00:00    41.0
4 2040-09-26 04:00:00    41.0
5 2040-09-26 05:00:00    41.0
6 2040-09-26 06:00:00    41.0
7 2040-09-26 07:00:00    41.0
8 2040-09-26 08:00:00    41.0
9 2040-09-26 09:00:00    41.0

Timestep dtype: datetime64[ns]
Weight dtype: float64

✓ CSV saved to: output/Final_Set/extracted_data/BL_H5_KM_weights.csv
✓ Total rows saved: 240
